# Import Libraries

In [1]:
# Если ещё не установлены:
%pip install -q numpy pandas torch datasets transformers sacrebleu sentencepiece protobuf
%pip install -e ../../

Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'e:\PycharmProjects\grand_lantern\grandlantern_venv\Scripts\python.exe -m pip install --upgrade pip' command.


Obtaining file:///E:/PycharmProjects/grand_lantern
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: still running...
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for grandlantern (pyproject.toml): started
  Building editable for grandlantern (pyproject.toml): finished with status 'done'
  Created wheel for grandlantern: filename=grandlantern-0.1.0-0.editable-py3-none-any.whl size=14523 sha256=95cfc7db6ac8f2ffb22da7c380f0b3db9c9899480790122a678c72ba986c3a74
  Stored in directory: C:\Users\mypc\AppData\Local\Temp\pip-ephem-wheel-cac

You should consider upgrading via the 'e:\PycharmProjects\grand_lantern\grandlantern_venv\Scripts\python.exe -m pip install --upgrade pip' command.


In [2]:
import numpy as np
import torch
from copy import copy
from datasets import load_dataset
from transformers import T5Tokenizer, AutoTokenizer
import sacrebleu

from grandlantern.matrix import Matrix
from grandlantern.layers import (
    Layer, LinearLayer, Linear, SoftMax,
    TransformerEncoderStack, TransformerDecoderStack,
)
from grandlantern.metrics import CrossEntropy, Metric
from grandlantern.optimizers import Adam, SGD
from grandlantern.models import EncDecTransformerModel
from grandlantern.generators import AutoRegressiveEncoderGenerator

e:\PycharmProjects\grand_lantern\grandlantern_venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Config

In [3]:
# --- Спецтокены T5 (в вашем фреймворке константы могут быть глобальными;
#     здесь переопределяем под токенизатор T5) ---
TOKEN_PAD = 0
TOKEN_EOS = 1
TOKEN_UNK = 2
TOKEN_BOS = 0            # T5 не имеет BOS; в качестве стартового токена — PAD
LIB_BOS = 2

# --- Данные ---
TRAIN_LIMIT = 8000
VAL_LIMIT   = 1000
TEST_LIMIT  = 200
MAX_SRC_LEN = 64
MAX_TGT_LEN = 64
BATCH_SIZE  = 8

# --- Модель ---
VOCAB     = 250100         # заполним после загрузки токенизатора
D_MODEL   = 256
N_HEADS   = 4
ATTN      = D_MODEL // N_HEADS
FFN       = 512
N_LAYERS  = 2
DROPOUT   = 0.1
N_EPOCHS  = 5
LR        = 3e-4

# Dataset Load

In [4]:
# Параллельный корпус Tatoeba en-ru
ds = load_dataset("xmarva/tfs-tatoeba-en-ru")
print(ds)

tokenizer = AutoTokenizer.from_pretrained("google/mt5-small")
VOCAB = tokenizer.vocab_size
print("vocab size:", VOCAB)
print("pad/bos/eos/unk:",
      tokenizer.pad_token_id,
      tokenizer.bos_token_id,
      tokenizer.eos_token_id,
      tokenizer.unk_token_id)

# Санити-чек: T5 использует PAD=0, EOS=1
assert tokenizer.pad_token_id == TOKEN_PAD
assert tokenizer.eos_token_id == TOKEN_EOS

DatasetDict({
    train: Dataset({
        features: ['en', 'ru'],
        num_rows: 508093
    })
    validation: Dataset({
        features: ['en', 'ru'],
        num_rows: 10370
    })
    test: Dataset({
        features: ['en', 'ru'],
        num_rows: 36385
    })
})


You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
e:\PycharmProjects\grand_lantern\grandlantern_venv\lib\site-packages\transformers\convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


vocab size: 250100
pad/bos/eos/unk: 0 None 1 2


In [5]:
def preprocess_split(split, limit=None):
    """
    Возвращает списки int-массивов: src_ids, src_mask, tgt_ids, tgt_mask.
    Маски пока не паддинуты до фиксированной длины — сделаем в итераторе,
    чтобы не тратить память.
    """
    if limit is not None:
        split = split.select(range(min(limit, len(split))))

    src_ids, src_masks, tgt_ids, tgt_masks = [], [], [], []

    for ex in split:
        # T5 требует префикс задачи для перевода
        src_text = "translate English to Russian: " + ex["en"]

        enc_src = tokenizer(
            src_text,
            max_length=MAX_SRC_LEN,
            truncation=True,
            padding=False,
            return_attention_mask=True,
            return_tensors="np",
        )
        enc_tgt = tokenizer(
            ex["ru"],
            max_length=MAX_TGT_LEN,
            truncation=True,
            padding=False,
            return_attention_mask=True,
            return_tensors="np",
        )

        src_ids.append(enc_src["input_ids"][0])
        src_masks.append(enc_src["attention_mask"][0])
        tgt_ids.append(enc_tgt["input_ids"][0])
        tgt_masks.append(enc_tgt["attention_mask"][0])

    return src_ids, src_masks, tgt_ids, tgt_masks


train_src, train_src_m, train_tgt, train_tgt_m = preprocess_split(ds["train"],      TRAIN_LIMIT)
val_src,   val_src_m,   val_tgt,   val_tgt_m   = preprocess_split(ds["validation"], VAL_LIMIT)
test_src,  test_src_m,  test_tgt,  test_tgt_m  = preprocess_split(ds["test"],       TEST_LIMIT)

print("train:", len(train_src), "val:", len(val_src), "test:", len(test_src))

train: 8000 val: 1000 test: 200


In [6]:
class TranslationIterator:
    """
    Выдаёт (X, X_mask, y, y_mask) как Matrix.
    Паддинг — до максимальной длины в батче.
    """
    def __init__(self, src_ids, src_masks, tgt_ids, tgt_masks,
                 batch_size=32, shuffle=True):
        self.src_ids   = src_ids
        self.src_masks = src_masks
        self.tgt_ids   = tgt_ids
        self.tgt_masks = tgt_masks
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.n_batches = (len(src_ids) + batch_size - 1) // batch_size

    def __copy__(self):
        return TranslationIterator(
            self.src_ids, self.src_masks, self.tgt_ids, self.tgt_masks,
            self.batch_size, self.shuffle,
        )

    def __call__(self):
        idx = np.arange(len(self.src_ids))
        if self.shuffle:
            np.random.shuffle(idx)

        for start in range(0, len(idx), self.batch_size):
            batch = idx[start:start + self.batch_size]

            # --- паддинг source ---
            max_src = max(len(self.src_ids[i]) for i in batch)
            X = np.zeros((len(batch), max_src), dtype=np.int64)
            X_mask = np.zeros_like(X)
            for j, i in enumerate(batch):
                L = len(self.src_ids[i])
                X[j, :L] = self.src_ids[i]
                X_mask[j, :L] = 1

            # --- паддинг target ---
            max_tgt = max(len(self.tgt_ids[i]) for i in batch)
            y = np.zeros((len(batch), max_tgt), dtype=np.int64)
            y_mask = np.zeros_like(y)
            for j, i in enumerate(batch):
                L = len(self.tgt_ids[i])
                y[j, :L] = self.tgt_ids[i]
                y_mask[j, :L] = 1

            yield Matrix(X), Matrix(X_mask), Matrix(y), Matrix(y_mask)


train_it = TranslationIterator(
    train_src, train_src_m, train_tgt, train_tgt_m,
    batch_size=BATCH_SIZE, shuffle=True,
)
val_it = TranslationIterator(
    val_src, val_src_m, val_tgt, val_tgt_m,
    batch_size=BATCH_SIZE, shuffle=False,
)
test_it = TranslationIterator(
    test_src, test_src_m, test_tgt, test_tgt_m,
    batch_size=BATCH_SIZE, shuffle=False,
)
print("train batches:", train_it.n_batches, "val:", val_it.n_batches)

train batches: 1000 val: 125


In [7]:
class GatherEmbedding(Layer):
    """W[indices] с корректным backward через scatter-add."""
    def __init__(self, vocab_size, d_model):
        super().__init__()
        self.W = Matrix(
            np.random.randn(vocab_size, d_model).astype(np.float32) * 0.02,
            require_grad=True,
        )
        self.parameters = [self.W]
        self.vocab_size = vocab_size
        self.d_model = d_model

    def forward(self, X, train_mode):
        idx = X.value.astype(int)                         # (B, T)
        flat = idx.reshape(-1)                            # (B*T,)
        selected = self.W.value[flat]                     # (B*T, d)
        new_value = selected.reshape(idx.shape + (self.d_model,))

        new_local_gradients = []
        if self.W.require_grad:
            def grad_W(grad):
                grad_flat = grad.reshape(-1, self.d_model)
                g = np.zeros_like(self.W.value)
                np.add.at(g, flat, grad_flat)             # scatter-add
                return g
            new_local_gradients.append((self.W, grad_W, "gather_embedding"))

        return Matrix(new_value, new_local_gradients, self.W.require_grad)

    def get_parameters(self):
        return self.parameters

In [8]:
class BLEUMetric(Metric):
    """
    BLEU на батче. Работает и с teacher-forcing (B,T,V), и с генерацией (B,T).
    Используется в существующем цикле модели (train/test).
    """
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    @staticmethod
    def _to_indices(x):
        v = x.value if hasattr(x, 'value') else np.asarray(x)
        if v.ndim == 3:                          # (B, T, V) → (B, T)
            v = v.argmax(axis=-1)
        return v.astype(int)

    def __call__(self, y_true, y_pred):
        refs = self._to_indices(y_true)
        hyps = self._to_indices(y_pred)

        ref_texts = [self.tokenizer.decode(s, skip_special_tokens=True) for s in refs]
        hyp_texts = [self.tokenizer.decode(s, skip_special_tokens=True) for s in hyps]
        return sacrebleu.corpus_bleu(hyp_texts, [ref_texts]).score

    def __str__(self):
        return "BLEU"


def corpus_bleu(model, iterator, tokenizer, max_batches=None):
    """
    Точный корпусный BLEU по всей валидации через генерацию.
    Вызывается раз в эпоху.
    """
    hyps, refs = [], []
    for i, (X, X_mask, y, y_mask) in enumerate(iterator()):
        if max_batches is not None and i >= max_batches:
            break
        y_gen = model.test_forward(X, X_mask)        # (B, T_gen) индексы
        for h in y_gen.value:
            hyps.append(tokenizer.decode(h, skip_special_tokens=True))
        for r in y.value:
            refs.append(tokenizer.decode(r, skip_special_tokens=True))
    return sacrebleu.corpus_bleu(hyps, [refs]).score, hyps[:3], refs[:3]

In [9]:
src_vec = GatherEmbedding(VOCAB, D_MODEL)
tgt_vec = GatherEmbedding(VOCAB, D_MODEL)

encoder = TransformerEncoderStack(
    n_layers=N_LAYERS, emb_dim=D_MODEL,
    attn_neurons=ATTN, n_heads=N_HEADS, ffn_neurons=FFN,
    dropout_prob=DROPOUT,
)
decoder = TransformerDecoderStack(
    n_layers=N_LAYERS, emb_dim=D_MODEL,
    attn_neurons=ATTN, n_heads=N_HEADS, ffn_neurons=FFN,
    dropout_prob=DROPOUT,
)
out_layer = LinearLayer(
    n_neurons=VOCAB, activation=SoftMax(), biased=True,
)

generator = AutoRegressiveEncoderGenerator(
    token_bos=LIB_BOS, token_eos=TOKEN_EOS,
    max_len=MAX_TGT_LEN, strategy="greedy",
)

model = EncDecTransformerModel(
    n_epochs=N_EPOCHS,
    dataset_iterator=train_it,
    loss_function=CrossEntropy(use_mask=True, use_one_hot=False),
    metric_function=BLEUMetric(tokenizer),
    optimizer=Adam(learning_rate=LR),
    src_vectorizer=src_vec,
    tgt_vectorizer=tgt_vec,
    encoder=encoder,
    decoder=decoder,
    out_layer=out_layer,
    generator=generator,
)
print("Model init")

Model init


In [10]:
import time

for epoch in range(N_EPOCHS):
    t0 = time.time()

    # --- обучение ---
    loss_train, bleu_tf_train = model.train(train_it)

    # --- валидация: teacher-forcing loss + generation BLEU ---
    loss_val, bleu_val = model.test(val_it)

    # --- точный корпусный BLEU на срезе валидации ---
    bleu_corpus, hyp_ex, ref_ex = corpus_bleu(model, val_it, tokenizer, max_batches=5)

    dt = time.time() - t0
    print(f"Epoch {epoch+1}/{N_EPOCHS} ({dt:.1f}s)")
    print(f"  train loss : {loss_train:.4f}    train BLEU(tf): {bleu_tf_train:.2f}")
    print(f"  val   loss : {loss_val:.4f}    val   BLEU(gen): {bleu_val:.2f}")
    print(f"  val   BLEU(corpus, 5 batches): {bleu_corpus:.2f}")
    print(f"  HYP: {hyp_ex[0][:90] if hyp_ex else ''}")
    print(f"  REF: {ref_ex[0][:90] if ref_ex else ''}")
    print("-" * 70)

print("Model fit. Params:", sum(p.value.size for p in model.parameters))

KeyboardInterrupt: 

In [12]:
# В ноутбуке, отдельная ячейка
X_one = Matrix(train_src[:1])
y_one = Matrix(train_tgt[:1])
Xm_one = Matrix(train_src_m[:1])
ym_one = Matrix(train_tgt_m[:1])

print("X:", X_one.value)
print("y:", y_one.value)
print("y_mask:", ym_one.value)

model.optimizer = Adam(learning_rate=3e-4)
for step in range(200):
    model.collect_params_reguls()
    model.zero_grad()
    y_pred = model.train_forward(X_one, y_one, Xm_one, ym_one)
    loss = model.loss_fn(y_one, y_pred, ym_one)
    loss.backward()
    model.optimizer.optimize(model.parameters)
    if step % 20 == 0:
        print(f"step {step}: loss={float(loss.value):.4f}")

# Проверка генерации
y_gen = model.test_forward(X_one, Xm_one)
print("GEN:", tokenizer.decode(y_gen.value[0], skip_special_tokens=True))
print("REF:", tokenizer.decode(y_one.value[0], skip_special_tokens=True))

X: [[37194  5413   288 26796   267  1662  3609   783 12004   345  1638   259
  38328   260     1]]
y: [[10285  8684 22808  2702 14193 16876   259  1802 16999   354   260     1]]
y_mask: [[1 1 1 1 1 1 1 1 1 1 1 1]]
step 0: loss=9.1747
step 20: loss=3.4192
step 40: loss=1.0009
step 60: loss=0.7767
step 80: loss=0.7688
step 100: loss=0.7688
step 120: loss=0.7685
step 140: loss=0.7683
step 160: loss=0.7683
step 180: loss=0.7680
GEN: бебебеу.
REF: Тебе стоило последовать его совету.
